# 到着ステーション別 平均利用時間の分析

出発ステーションをドロップダウンで選ぶと、到着ステーション別の平均利用時間がグラフで表示されます。

## 1. データの取得

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings

plt.rcParams['font.family'] = 'MS Gothic'
plt.rcParams['figure.figsize'] = (14, 7)
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

client = bigquery.Client(project='samplestudy-465703')
print('準備完了！')

BigQuery から「出発ステーション × 到着ステーション」ごとの平均利用時間とトリップ数を取得します。

信頼性のあるデータにするため、**10回以上の利用があるペア**に絞っています。

In [ ]:
query = """
SELECT
    start_station_name,
    end_station_name,
    ROUND(AVG(duration_minutes), 1) AS avg_duration,
    COUNT(*) AS trip_count
FROM `bigquery-public-data.austin_bikeshare.bikeshare_trips`
WHERE start_station_name IS NOT NULL
    AND end_station_name IS NOT NULL
    AND duration_minutes IS NOT NULL
    AND duration_minutes > 0
GROUP BY start_station_name, end_station_name
HAVING COUNT(*) >= 10
ORDER BY start_station_name, avg_duration DESC
"""

df = client.query(query).to_dataframe()
print(f'取得完了: {len(df):,} 件（出発 {df["start_station_name"].nunique()} 駅 × 到着 {df["end_station_name"].nunique()} 駅）')

## 2. インタラクティブ分析

下のドロップダウンから出発ステーションを選ぶと、そこから出発したトリップの到着先別・平均利用時間がグラフで表示されます。

In [ ]:
# 出発ステーションをトリップ数が多い順に並べる（選びやすいように）
station_trip_counts = (
    df.groupby('start_station_name')['trip_count']
    .sum()
    .sort_values(ascending=False)
)
station_list = station_trip_counts.index.tolist()

# --- ウィジェット（UI部品）の作成 ---
station_dropdown = widgets.Dropdown(
    options=station_list,
    value=station_list[0],
    description='出発駅:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='450px'),
)

top_n_slider = widgets.IntSlider(
    value=15, min=5, max=30, step=5,
    description='表示件数:',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='300px'),
)

output = widgets.Output()

def update_chart(change=None):
    with output:
        clear_output(wait=True)
        station = station_dropdown.value
        top_n = top_n_slider.value

        filtered = (
            df[df['start_station_name'] == station]
            .sort_values('avg_duration', ascending=False)
            .head(top_n)
        )

        if filtered.empty:
            print(f'「{station}」からのデータがありません')
            return

        total_trips = filtered['trip_count'].sum()

        fig, ax = plt.subplots(figsize=(14, max(5, top_n * 0.4)))

        colors = sns.color_palette('viridis', len(filtered))
        bars = ax.barh(
            range(len(filtered)),
            filtered['avg_duration'],
            color=colors,
        )
        ax.set_yticks(range(len(filtered)))
        ax.set_yticklabels(filtered['end_station_name'], fontsize=10)
        ax.invert_yaxis()
        ax.set_xlabel('平均利用時間（分）', fontsize=12)
        ax.set_title(
            f'「{station}」発 → 到着ステーション別 平均利用時間\n'
            f'（対象: {total_trips:,} トリップ）',
            fontsize=14, pad=15,
        )

        for bar, row in zip(bars, filtered.itertuples()):
            ax.text(
                bar.get_width() + 0.5,
                bar.get_y() + bar.get_height() / 2,
                f'{row.avg_duration:.1f}分 ({row.trip_count:,}件)',
                va='center', fontsize=9, color='#333',
            )

        ax.set_xlim(0, filtered['avg_duration'].max() * 1.3)
        plt.tight_layout()
        plt.show()

station_dropdown.observe(update_chart, names='value')
top_n_slider.observe(update_chart, names='value')

display(widgets.HBox([station_dropdown, top_n_slider]))
display(output)
update_chart()